# 5.8 MNIST 测试实战

前面已经介绍并实现了前向传播和数据集的加载部分，现在我们来完成剩下的分类任务逻辑。在训练的过程中，通过间隔数个 Step 后打印误差数据，可以有效监督模型的训练
进度，代码如下

In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import datasets  # 导入经典数据集加载模块

(x, y), (x_test, y_test) = datasets.mnist.load_data()  # 加载 MNIST 数据集
print('x.shape', x.shape, 'y_shape', y.shape,
      'x_test', x_test.shape, 'y_test', y_test.shape)

train_db = tf.data.Dataset.from_tensor_slices((x, y))  # 构建 Dataset 对象
train_db.shuffle(buffer_size=10000)  # 随机打散样本，不会打乱样本与标签映射关系
# buffer_size 参数指定缓冲池的大小，一般设置为一个较大的常数即可

train_db.batch(128)  # 设置批训练，batch size 为 128

def preprocess(x, y):  # 自定义的预处理函数
    # 调用此函数时会自动传入 x,y 对象，shape 为[b, 28, 28], [b]
    # 标准化到 0~1
    x = tf.cast(x, dtype=tf.float32) / 255.
    x = tf.reshape(x, [-1, 28 * 28])  # 打平
    y = tf.cast(y, dtype=tf.int32)  # 转成整型张量
    y = tf.one_hot(y, depth=10)  # one_hot
    return x, y


# 预处理函数实现在 preprocess 函数中，传入函数名即可
train_db = train_db.map(preprocess)


x.shape (60000, 28, 28) y_shape (60000,) x_test (10000, 28, 28) y_test (10000,)


In [ ]:

# 每层的张量都需要被优化，故使用 Variable 类型，并使用截断的正态分布初始化权值张量
# 偏置向量初始化为 0 即可
# 第一层的参数

w1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
b1 = tf.Variable(tf.zeros(256))

# 第二层参数
w2 = tf.Variable(tf.random.truncated_normal([256, 128], stddev=0.1))
b2 = tf.Variable(tf.zeros([128]))

# 第三层参数
w3 = tf.Variable(tf.random.truncated_normal([128, 10], stddev=0.1))
b3 = tf.Variable(tf.zeros([10]))

In [ ]:

out = tf.Variable()
total_correct = tf.Variable()
for epoch in range(20):
    for step, (x, y) in enumerate(train_db):
        h1 = x @ w1 + b1
        h1 = tf.nn.relu(h1)

        h2 = h1 @ w2 + b2
        h2 = tf.nn.relu(h2)

        out = h2 @ w3 + b3  # 预测值 out 的 shape 为[𝑏, 10]，分别代表了样本属于每个类别的概率

        loss = tf.square(y - out)
        loss = tf.reduce_mean(loss)

        if step % 50 == 0:
            print('loss:', loss)
        # 根据 tf.argmax 函数选出概率最大值出现的索引号，也即样本最有可能的类别号
        pred = tf.argmax(out, axis=1)
        # 由于我们的标注 y 已经在预处理中完成了 one-hot 编码，这在测试时其实是不需要的，因此通过 tf.argmax 可以得到数字编码的标注 y
        y = tf.argmax(y, axis=1)  # one-hot 编码逆过程
        correct = tf.equal(pred, y)  # 比较预测值与真实值
        total_correct += tf.reduce_sum(tf.cast(correct,
                                       dtype=tf.int32)).numpy()
# print(step, 'Evaluate Acc:', total_correct/total)